# Multi-Agent Orchestration in LangGraph

Module 01 built a single agent — one `agent` node, one `tools` node, one system prompt covering everything it needs to know. That works until the domain gets big enough that one system prompt has to juggle too much (HR policy *and* IT policy *and* expense rules *and* who knows what else), at which point accuracy on any single topic tends to degrade. The fix: split into **specialist agents**.

But "split into specialists" isn't one technique — it's five, and they're not interchangeable. This notebook builds four of them for real (the fifth, Swarm, is explained but not built — it's the same mechanism as Handoffs with the hub removed) and verifies each against current docs.langchain.com content, not assumed from memory:

1. **Router** — a classifier step dispatches each question to the right specialist, one-shot, no memory of its own
2. **Handoffs** — an agent decides *for itself* to transfer control to a specialist, via a tool call
3. **Subagents** — a main agent calls specialists *as tools* and stays in charge; they report back, they don't take over
4. **Skills** — no second agent at all; one agent loads specialized context on demand
5. **Human-in-the-loop** — pausing any of the above for a real approval step before a sensitive action executes

Then: shared vs. private state across agents, the Reflection/Reflexion pattern (a genuinely different loop from ReAct), and a closing decision guide for when to reach for which of the five.

Domain throughout: your company-policy documents, split into two specialists — an **HR Agent** (vacation, remote work) and an **IT/Finance Agent** (mobile phone policy, expenses). Same policy content you've used since `Build_Smarter_AI_Apps`.

## Setup

In [21]:
# pip install -U langgraph langchain langchain-anthropic langchain-chroma langchain-huggingface

import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]
print("✅ API key set.")

from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import AIMessage, HumanMessage
print("Imports OK")

✅ API key set.
Imports OK


### Sample knowledge base — split by domain

Same style of inline sample text as your other RAG notebooks (self-contained, no external download), but split into two separate small document sets — one per specialist — since each agent should only retrieve from its own domain.

In [22]:
hr_docs_text = """
Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval.
Requests must be submitted at least 48 hours in advance through the HR portal.

Vacation Policy: Full-time employees accrue 15 vacation days per year, accrued monthly.
Unused vacation days roll over up to a maximum of 5 days into the following year.
"""

it_finance_docs_text = """
Mobile Phone Policy: Company-issued phones may be used for personal calls within reason.
Employees must report a lost or stolen device to IT within 24 hours.

Expense Policy: Business expenses under $50 do not require pre-approval but must be
submitted with receipts within 30 days. Expenses over $50 require manager sign-off
before the purchase is made.
"""
print("Sample knowledge base ready")

Sample knowledge base ready


### Real retrieval tools per specialist (same Chroma pattern as `Build_Smarter_AI_Apps`)

Each specialist gets its own small vector store, built the same way you already know.

In [24]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.tools import tool

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)

hr_chunks = splitter.create_documents([hr_docs_text])
hr_vectorstore = Chroma.from_documents(hr_chunks, embedding_model, collection_name="hr_policies")

it_finance_chunks = splitter.create_documents([it_finance_docs_text])
it_finance_vectorstore = Chroma.from_documents(it_finance_chunks, embedding_model, collection_name="it_finance_policies")

@tool
def retrieve_hr_policy(query: str) -> str:
    """Retrieve relevant HR policy context (vacation, remote work)."""
    docs = hr_vectorstore.similarity_search(query, k=2)
    return "\n\n".join(d.page_content for d in docs)

@tool
def retrieve_it_finance_policy(query: str) -> str:
    """Retrieve relevant IT/Finance policy context (mobile phones, expenses)."""
    docs = it_finance_vectorstore.similarity_search(query, k=2)
    return "\n\n".join(d.page_content for d in docs)

print("Retrieval tools defined")

Retrieval tools defined


**On `@tool` vs. `.as_retriever()`:** these are two separate, unrelated mechanisms, worth not conflating. `.as_retriever()` (`Build_Smarter_AI_Apps` §8) wraps a **vectorstore** into a `VectorStoreRetriever` Runnable — not used anywhere in this notebook. `@tool` wraps a **plain Python function** — here, `retrieve_hr_policy`, whose *body* happens to call `hr_vectorstore.similarity_search(...)` directly — into an LLM-callable tool with an auto-generated schema. `hr_vectorstore` itself is never converted into anything; it's just called directly, inside an ordinary function, and that function is what gets the `@tool` treatment.

## 1. Router — classify, then dispatch

Confirmed against docs.langchain.com's current multi-agent patterns page: Router is *"a routing step [that] classifies input and directs it to one or more specialized agents"* — specifically **stateless**, no memory of its own across turns; that's what distinguishes it from Subagents (§3), which does maintain conversation state across multiple turns. So it doesnt return a message to the list. 

The simplest version: a `supervisor` node that doesn't change state at all, just exists so a router function can run after it; the router calls the model to classify which specialist should handle the question, and `add_conditional_edges` sends execution to the matching node. Each specialist here is `create_agent` from Module 01, given only its own retrieval tool.

In [4]:
from langchain.agents import create_agent
from typing import Literal

hr_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[retrieve_hr_policy],
    system_prompt="You answer HR policy questions (vacation, remote work) using the retrieval tool. Say you don't know if the context doesn't cover it.",
    name="hr_agent",
)

it_finance_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[retrieve_it_finance_policy],
    system_prompt="You answer IT/Finance policy questions (mobile phones, expenses) using the retrieval tool. Say you don't know if the context doesn't cover it.",
    name="it_finance_agent",
)

from langchain_anthropic import ChatAnthropic
classifier_model = ChatAnthropic(model="claude-haiku-4-5", max_tokens=20, temperature=0)

def supervisor_node(state: MessagesState) -> dict:
    return {}  # no state change -- this node exists purely so the router below can run after it

def route_to_specialist(state: MessagesState) -> Literal["hr_agent", "it_finance_agent"]:
    question = state["messages"][-1].content
    decision = classifier_model.invoke(
        f"Classify this question as exactly 'hr_agent' or 'it_finance_agent', nothing else.\nQuestion: {question}"
    )
    return decision.content.strip()

builder_router = StateGraph(MessagesState)
builder_router.add_node("supervisor", supervisor_node)
builder_router.add_node("hr_agent", hr_agent)
builder_router.add_node("it_finance_agent", it_finance_agent)
builder_router.add_edge(START, "supervisor")
builder_router.add_conditional_edges("supervisor", route_to_specialist, {"hr_agent": "hr_agent", "it_finance_agent": "it_finance_agent"})
builder_router.add_edge("hr_agent", END)
builder_router.add_edge("it_finance_agent", END)

router_graph = builder_router.compile()
print("Router graph compiled (construction only, no API call yet).")

Router graph compiled (construction only, no API call yet).


**⚠️ Needs your API key** — the cell below actually classifies and routes a real question.

In [ ]:
result = router_graph.invoke({"messages": [HumanMessage(content="How many vacation days do I get?")]})
print(result["messages"][-1].content)

### 🔍 Under the hood — why `supervisor_node` does nothing, and the graph-within-a-graph

`supervisor_node` returns `{}` — no state update at all. It exists purely as an **anchor point** for `add_conditional_edges` to attach to. If you wanted the supervisor to actually do something (log the question, extract metadata) beyond routing, that logic would go inside this node function, with the routing decision itself still made by the separate router function. Per LangGraph's own guidance: router functions stay pure (read state, return a string), real computation belongs in nodes.

Worth being concrete about what `builder_router.add_node("hr_agent", hr_agent)` actually registers: `create_agent(...)` returns a `CompiledStateGraph` — a complete agent/tools loop with its own internal `MessagesState`, `agent` node, `ToolNode`, `tools_condition`. `add_node` doesn't care whether what you hand it is a plain function or a whole other graph, only that it behaves like a `Runnable`. So when `router_graph.invoke(...)` reaches the `"hr_agent"` node, that's one step from the *outer* graph's point of view, but underneath it's `hr_agent.invoke(current_state)` running its own internal loop (decide to call `retrieve_hr_policy`, run the tool, come back, answer) — invisible to the outer graph, which only sees the final combined state update.

**What actually shapes `hr_agent`'s answer, precisely:** two things combine — its `system_prompt` (*"You answer HR policy questions... using the retrieval tool. Say you don't know if the context doesn't cover it."*) and whatever `retrieve_hr_policy` actually returns that turn (the real vacation-policy text pulled from `hr_vectorstore`). The system prompt tells Claude its role and to ground answers in the tool's results; a phrasing like "Based on our HR policy, here's what..." is just Claude's own natural way of presenting an answer once told to answer *from* retrieved policy content — not a literal quote from the system prompt, a direct consequence of it.

**One shape worth expecting when you run this for real:** `hr_agent`/`it_finance_agent` use `claude-sonnet-4-6`, a chattier model than the one-word `classifier_model` (`claude-haiku-4-5`). It's fairly common for it to narrate what it's about to do ("Let me look that up for you!") *in the same response* where it also decides to call a tool — meaning `AIMessage.content` can be a list containing **both** a text block and a `tool_use` block at once, not one or the other. `classifier_model` never shows this because it's explicitly told to respond with exactly one word, leaving no room for extra text. Whichever specialist answers, that narration text is its own reasoning as it processes the original user question inside its own internal agent/tools loop — not a response to some special internal prompt, and not the supervisor's doing.

## 2. Handoffs, done right

Confirmed against docs.langchain.com's current Handoffs page: the mechanism is tool calls updating state to trigger routing — *"tools update a state variable... and the system reads this variable to adjust behavior... or routing to a different agent."* The `Command` object is how LangGraph implements that update-and-route-in-one-step.

`Command(goto=target, update={...})` is a single object that does two jobs a plain node + router pair used to need two separate mechanisms for: it tells LangGraph *where to go next* and *what to change in state*, together. You'll see its exact shape in the real code below — the line `return Command(goto=agent_name, graph=Command.PARENT, update={...})` inside `handoff()`. `graph=Command.PARENT` is the piece that matters once agents become subgraphs nested inside a larger graph (exactly the situation here): a node *inside* one agent's subgraph can hand off directly to a *different* agent subgraph elsewhere in the parent graph, which a plain conditional edge (scoped only to its own graph) cannot do at all.

**Worth noticing in `builder_handoffs` below, by contrast with `builder_router` above:** there's no `add_conditional_edges` call anywhere. With plain edges (§1), the destination has to be declared upfront in the graph structure. With `Command(goto=target, ...)`, the routing decision is computed dynamically *inside the node's own return value* at runtime — no separate edge declaration needed for it at all; `supervisor_agent` decides where to go as part of doing its own work, rather than a router reading its output afterward.

**And to be explicit about what didn't change from §1:** this is still the exact same real RAG retrieval — `hr_agent`/`it_finance_agent` are the identical `create_agent` instances built in §1, same `retrieve_hr_policy`/`retrieve_it_finance_policy` tools and all. Handoffs only changes *how the supervisor routes*, not what happens once a specialist has control.

**Where this lives matters as much as what it is.** The documented pattern is *tool-initiated*: a specialist agent calls a tool, and that tool's own code is what returns the `Command`. Verified directly against `langgraph-supervisor`'s installed source (`langgraph_supervisor/handoff.py`, `create_handoff_tool`) — this is a simplified version of exactly what that function does internally, with the parallel-tool-call edge case removed for clarity.

**One Python-mechanics note before the code, since it's easy to trip on otherwise:** `make_handoff_tool(agent_name)` is a *factory function* — it doesn't return a value, it returns a whole new `@tool`-decorated function each time it's called. That's why it's written this way rather than just writing `transfer_to_hr_agent` and `transfer_to_it_finance_agent` directly: `@tool` needs a fixed function body to decorate, but the two tools are identical except for which `agent_name` they hand off to, so `make_handoff_tool` exists purely to stamp out that one difference twice without duplicating the whole function.

In [26]:
# Pure Python, zero LangChain -- the exact same closure pattern make_handoff_tool
# uses below, isolated so the mechanic is visible on its own before @tool joins in.
def make_greeter(name):
    def greet():
        return f"Hello, {name}!"
    return greet

say_hi_to_sarah = make_greeter("Sarah")
say_hi_to_bob = make_greeter("Bob")

print(say_hi_to_sarah())
print(say_hi_to_bob())
# Two DIFFERENT function objects came out of make_greeter -- each one "remembers"
# its own `name` from the moment it was created. make_handoff_tool does exactly
# this, just with @tool decorating the inner function instead of leaving it plain.

Hello, Sarah!
Hello, Bob!


In [ ]:
from typing import Annotated
from langchain_core.tools import InjectedToolCallId
from langgraph.prebuilt import InjectedState
from langchain_core.messages import ToolMessage
from langgraph.types import Command


def make_handoff_tool(agent_name: str):
    """Builds a transfer_to_<agent_name> tool -- an agent calls this itself
    to hand off, rather than a central classifier deciding on its behalf."""

    @tool(
        f"transfer_to_{agent_name}",
        description=f"Transfer the conversation to {agent_name} for questions in its domain.",
    )
    def handoff(
        state: Annotated[dict, InjectedState],              # the calling agent's own state, injected automatically
        tool_call_id: Annotated[str, InjectedToolCallId],    # this specific tool call's id, injected automatically
    ) -> Command:
        # Every tool_use needs a matching tool_result or the message history is invalid --
        # this ToolMessage IS that tool_result, using the injected tool_call_id.
        tool_message = ToolMessage(
            content=f"Successfully transferred to {agent_name}",
            name=f"transfer_to_{agent_name}",
            tool_call_id=tool_call_id,
        )
        return Command(
            goto=agent_name,
            graph=Command.PARENT,  # jump to a node in the graph that called this one
            update={"messages": state["messages"] + [tool_message]},    # add teh toolMessage to the state list
        )

    return handoff  # return an instance of the function unrun. 


transfer_to_hr_agent = make_handoff_tool("hr_agent")    # NOTE its handing an instance of the handoff function unrun. I think??
transfer_to_it_finance_agent = make_handoff_tool("it_finance_agent")

supervisor_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[transfer_to_hr_agent, transfer_to_it_finance_agent],
    system_prompt=(
        "You are a supervisor. Decide whether the question is HR-related (vacation, remote work) "
        "or IT/Finance-related (mobile phones, expenses), then call the matching transfer tool."
    ),
    name="supervisor",
)

builder_handoffs = StateGraph(MessagesState)
builder_handoffs.add_node("supervisor", supervisor_agent)
builder_handoffs.add_node("hr_agent", hr_agent)
builder_handoffs.add_node("it_finance_agent", it_finance_agent)
builder_handoffs.add_edge(START, "supervisor")
builder_handoffs.add_edge("hr_agent", END)
builder_handoffs.add_edge("it_finance_agent", END)

handoffs_graph = builder_handoffs.compile()
print("Handoffs graph compiled (construction only, no API call yet).")

# so the agent/ttool loop still exists because tthe supervisor is built with create agent and therefore has that in the subgraph
# except it was broken because the supervisor handed it off. Then again the agent/tool loop exists again because the specialist was also created with create agent?

Handoffs graph compiled (construction only, no API call yet).


### 🔍 Under the hood — this is the densest code in the notebook, so slow down here

Two things are genuinely new in `handoff()` — not restatements of anything from Module 01 — worth separating out.

**1. What `InjectedState`/`InjectedToolCallId` actually do.** Every tool you've built before this (`calculator`, `retrieve_hr_policy`) had parameters the *LLM* fills in — Claude decides the `query` string, and that becomes part of the tool-use request. `state` and `tool_call_id` here are different in kind, not just in name: wrapping a parameter's type hint in `Annotated[..., InjectedState]` or `Annotated[..., InjectedToolCallId]` tells `@tool` to **exclude that parameter from the schema Claude ever sees**. If you inspected `transfer_to_hr_agent`'s schema directly, it would show *zero* parameters — Claude calls it with `args={}`, deciding only *that* it should hand off, never *what* to hand off, because there's nothing left for it to decide. Then, when `ToolNode` actually executes the call, it recognizes these two specific annotations and fills them in itself: `state` becomes whatever the calling agent's current state actually is, and `tool_call_id` becomes the real id off the specific `tool_use` block that triggered this call — both sourced from the graph's own runtime, never from the model.

Why bother, rather than just asking Claude to pass these as normal arguments? Two concrete reasons this code depends on: `state["messages"]` needs to be the *entire real conversation so far*, not a summary Claude reconstructs from memory (which could drop or garble something); and `tool_call_id` has to be an *exact* match to the triggering tool call or the resulting `ToolMessage` won't pair correctly and the message history becomes invalid — not the kind of value you want an LLM copying by hand when the framework already knows it exactly.

**2. What `graph=Command.PARENT` actually resolves to, in this exact code — not in the abstract.** First, in plain terms, since it's easy to misread "PARENT" as something to do with class inheritance — it isn't. Think of graphs as folders that can contain other folders: `builder_handoffs` is the outer folder. `supervisor_agent` isn't a plain file sitting in that folder, it's a *whole other folder* (a complete compiled graph, dropped in as `builder_handoffs`'s `"supervisor"` node — same "graph within a graph" idea from §1). `handoff()` lives one folder deeper still — inside `supervisor_agent`'s *own* internal tools node. So from `handoff()`'s point of view, "the graph I'm running inside" is that small inner folder, which only ever contains `"agent"`/`"tools"` — the fixed shape `create_agent` always builds — there's no `"hr_agent"` folder down there at all. `graph=Command.PARENT` is the explicit instruction "look one folder up from where I am" — out of `supervisor_agent`'s tiny inner graph, into `builder_handoffs`, where `"hr_agent"` genuinely exists as a node. Leave it out, and LangGraph searches the wrong (inner) folder and can't find the destination.

There are three graphs in play here, nested: `builder_handoffs` (the outermost graph you build and compile below), `supervisor_agent` (itself a compiled subgraph, built by `create_agent(...)`, registered as `builder_handoffs`'s `"supervisor"` node), and `handoff()` — a tool living *inside* `supervisor_agent`'s own internal `ToolNode`, one level further in still. This is exactly the situation flagged as needing `graph=Command.PARENT` back when `Command` was first introduced — you're seeing the concrete case that abstract description was describing.

**⚠️ Needs your API key** — the cell below should produce: `supervisor_agent` calls `transfer_to_hr_agent`, control jumps to `hr_agent`, `hr_agent` answers using `retrieve_hr_policy`. Print `result["messages"]` afterward to see the real shape: an `AIMessage` with `tool_calls=[{"name": "transfer_to_hr_agent", ...}]`, a `ToolMessage`, then `hr_agent`'s answer — the routing decision is a genuine tool call, same mechanism as any other tool call in this curriculum.

In [28]:
result = handoffs_graph.invoke({"messages": [HumanMessage(content="How many vacation days do I get?")]})
print(result["messages"][-1].content)

Based on our HR policy, here's what you need to know about vacation days:

- **Full-time employees** receive **15 vacation days per year**, which are accrued monthly.
- **Unused vacation days** can roll over into the following year, but only up to a maximum of **5 days**.

If you have any other questions or need more details, feel free to ask!


In [37]:
for m in result['messages']:
    print('\n',m.type)
    print('\n',m.content)
    print('-'*6)


 human

 How many vacation days do I get?
------

 ai

 [{'id': 'toolu_01P38UH45D5XATXGhU3v1qSD', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'transfer_to_hr_agent', 'type': 'tool_use'}]
------

 tool

 Successfully transferred to hr_agent
------

 ai

 [{'text': 'Let me look that up for you!', 'type': 'text'}, {'id': 'toolu_01UWGasyh3hWa2wMKdVnz4Dv', 'caller': {'type': 'direct'}, 'input': {'query': 'number of vacation days'}, 'name': 'retrieve_hr_policy', 'type': 'tool_use'}]
------

 tool

 Vacation Policy: Full-time employees accrue 15 vacation days per year, accrued monthly.
Unused vacation days roll over up to a maximum of 5 days into the following year.

Vacation Policy: Full-time employees accrue 15 vacation days per year, accrued monthly.
Unused vacation days roll over up to a maximum of 5 days into the following year.
------

 ai

 Based on our HR policy, here's what you need to know about vacation days:

- **Full-time employees** receive **15 vacation days per year**,

### 🔗 Ties back to theory — this is the pattern to actually internalize

Tool-initiated (an agent decides to call a transfer tool, same as any other tool call), using `Command(goto=..., graph=Command.PARENT)`, with full visibility into and control over exactly what gets passed to each specialist via `update={"messages": ...}` — that last part is "context engineering," and it's the specific control you'd lose by reaching for a prebuilt package instead of hand-rolling this. (Worth knowing as trivia, not something to run: `langgraph-supervisor`'s `create_supervisor()` builds this *exact* mechanism for you automatically, generating `transfer_to_*` tools the same shape as `make_handoff_tool` above — but its own current docs now recommend hand-writing this pattern directly for new production work rather than reaching for the package, which is why it's not built as its own section here.)

**Worth tracing exactly which message is whose, once you run this for real:** `handoff()`'s `update` is `{"messages": state["messages"] + [tool_message]}` — so the *only* thing the handoff itself contributes to the conversation is that one `ToolMessage` (`"Successfully transferred to hr_agent"`), which exists purely because a `tool_use` block always needs a matching `tool_result` or the message history becomes invalid — not because it has anything meaningful to say. Compare that to `supervisor_node` in §1, which returns `{}` and contributes literally nothing: same "the routing step's own footprint is minimal" idea, just concretely nonzero here because tool calls carry that extra bookkeeping requirement. Everything after that `ToolMessage` in the final `result["messages"]` — the actual answer — is `hr_agent`'s own output once it took over, not the supervisor's.

## 3. Subagents — a main agent calls specialists as tools, and stays in charge

Confirmed against docs.langchain.com: *"A main agent coordinates subagents as tools. All routing passes through the main agent, which decides when and how to invoke each subagent."* Key characteristics also confirmed from the same page: **no direct user interaction** (subagents return results to the main agent, not the user), and **parallel execution** (the main agent can invoke multiple subagents in a single turn).

**The precise contrast with Handoffs, since they're easy to blur together:** in Handoffs, control genuinely *transfers* — the specialist takes over, and the original agent is out of the loop unless the specialist hands back. In Subagents, control never leaves the main agent — a specialist runs, returns one result, and the main agent keeps going, exactly like any other tool call. Use Subagents when the main agent should stay the one talking to the user and just farm out a self-contained task; use Handoffs when a genuinely different agent should take over actually answering.

Building this requires no new mechanism — `hr_agent`/`it_finance_agent` already exist; the only new piece is a thin `@tool` wrapper around each one's `.invoke()`:

so is it safe to say use this when memory is used?

In [ ]:

# so these are now a ttool within a tool? because the agent specialists also have a tool attached. 
# also note it looks like these just return text?
@tool
def consult_hr_agent(question: str) -> str:
    """Consult the HR specialist agent for questions about vacation, remote work, etc. Returns its answer."""
    result = hr_agent.invoke({"messages": [HumanMessage(content=question)]})
    return result["messages"][-1].content

@tool
def consult_it_finance_agent(question: str) -> str:
    """Consult the IT/Finance specialist agent for questions about mobile phones, expenses, etc. Returns its answer."""
    result = it_finance_agent.invoke({"messages": [HumanMessage(content=question)]})
    return result["messages"][-1].content

main_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[consult_hr_agent, consult_it_finance_agent],
    system_prompt=(
        "You are the main assistant. For HR questions, consult the HR agent. For IT/Finance "
        "questions, consult the IT/Finance agent. If a question spans both, consult both and "
        "combine their answers yourself."
    ),
    name="main_agent",
)
print("Subagents-style main_agent constructed (construction only, no API call yet).")

Subagents-style main_agent constructed (construction only, no API call yet).


**⚠️ Needs your API key** — try a question that spans both domains (e.g. *"What's the vacation policy and can I use my personal phone for work calls?"*) to see the "parallel execution" characteristic for real: a capable model can call `consult_hr_agent` and `consult_it_finance_agent` in the *same* turn, then synthesize both results into one answer — something Handoffs structurally cannot do, since control can only be in one specialist at a time.

In [39]:
result = main_agent.invoke({"messages": [HumanMessage(
    content="What's the vacation policy, and can I use my personal phone for work calls?"
)]})
print(result["messages"][-1].content)

Here's a combined summary of both answers:

---

### 🏖️ Vacation Policy
- **Accrual:** Full-time employees earn **15 vacation days per year**, accrued monthly.
- **Rollover:** Unused days can carry over to the next year, but are **capped at 5 days**.

---

### 📱 Personal Phone for Work Calls
The current policy doesn't explicitly address using a **personal phone for work calls**. Here's what is known:
- **Company-issued phones** may be used for personal calls within reason.
- Lost or stolen company devices must be **reported to IT within 24 hours**.

Since there's no clear policy on personal phone use for work, it's best to **check directly with your IT or HR department** for clarification — especially if you're wondering about reimbursement or data/security requirements.

---

Let me know if you have any other questions! 😊


### 🔍 Under the hood — why this needed zero new LangGraph mechanism

`consult_hr_agent` is just a Python function whose body happens to call `hr_agent.invoke(...)` instead of `hr_vectorstore.similarity_search(...)` — structurally identical to `retrieve_hr_policy` all the way back in Setup, just with a whole agent inside instead of a vector search. `@tool` doesn't know or care that the function it's wrapping happens to run a full agent loop internally; from `main_agent`'s perspective, `consult_hr_agent` is exactly as opaque as `calculator` was in Module 01 — call it, get a string back, keep going. This is the entire mechanism: Subagents isn't a new LangGraph feature, it's the ordinary tool-calling loop, recursively.

## 4. Skills — no second agent at all

**Note on why this is in a *multi-agent* notebook at all:** it technically isn't multi-agent — flagging this explicitly so it doesn't read as a mistake. docs.langchain.com groups Skills alongside Router/Handoffs/Subagents/Custom-workflow under one "multi-agent patterns" page anyway, because it's the answer to the same underlying question ("my one system prompt is juggling too much") even though its actual mechanism involves zero additional agents. Kept here, in this notebook, for that reason — but worth remembering it's a single-agent technique when you're deciding which pattern to reach for.

Confirmed against docs.langchain.com: *"Specialized prompts and knowledge loaded on-demand. A single agent stays in control while loading context from skills as needed."* This is a genuinely different axis from everything above — not a different way of coordinating multiple agents, but a way of avoiding needing a second agent in the first place.

Minimal real implementation: one `load_skill` tool, and a system prompt that tells the agent what's available without loading the full content upfront (so those tokens are only spent when actually needed):

In [7]:
SKILLS = {
    "hr_policy_expert": (
        "Detailed HR policy knowledge: remote work requires manager approval and 48 hours "
        "notice; vacation accrues 15 days/year, rolling over up to 5 days."
    ),
    "it_finance_expert": (
        "Detailed IT/Finance policy knowledge: lost devices must be reported within 24 hours; "
        "expenses under $50 need no pre-approval, over $50 needs manager sign-off."
    ),
}

@tool
def load_skill(skill_name: str) -> str:
    """Load detailed knowledge for a specific skill.
    Available skills: hr_policy_expert, it_finance_expert."""
    return SKILLS.get(skill_name, f"Unknown skill: {skill_name}")

skills_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[load_skill],
    system_prompt=(
        "You are a company assistant. You have access to two skills via the load_skill tool: "
        "hr_policy_expert and it_finance_expert. Load whichever is relevant before answering, "
        "rather than guessing."
    ),
    name="skills_agent",
)
print("Skills agent constructed (construction only, no API call yet).")

Skills agent constructed (construction only, no API call yet).


**⚠️ Needs your API key.**

In [ ]:
result = skills_agent.invoke({"messages": [HumanMessage(content="How many vacation days do I get?")]})
print(result["messages"][-1].content)

### 🔗 Ties back to theory — the real decision between Subagents, Handoffs, and Skills

All three solve "one system prompt is juggling too much," but at different costs:
- **Skills** — cheapest. One agent, one extra tool, no second `create_agent` call, no orchestration graph at all. Right whenever the underlying task genuinely is *one job with several modes* (a coding assistant that loads a "SQL mode" or "Python mode") rather than genuinely different jobs needing genuinely different expertise.
- **Subagents** — one step up. Real second agents exist, but the main agent stays fully in charge and can run them in parallel. Right when a task is self-contained enough to farm out and get a single result back.
- **Handoffs** — most structural. Control genuinely moves. Right when a specialist should *become* the one talking, not just report back — this HR/IT split is a Handoffs-shaped problem specifically because a user asking a vacation question wants the HR agent's full ongoing attention, not a one-shot consult.

This company-policy domain has been used as a Handoffs example throughout §2 for a reason — it's genuinely the best fit of the three for this specific case. Skills and Subagents were worth building anyway so you'd recognize when a *different* problem calls for one of them instead.

question!! So with subagents and handoffs were not really actually building graphs anymore?

## 5. Human-in-the-loop — pausing for real approval before a sensitive action

Confirmed against docs.langchain.com's current Human-in-the-loop guide: `HumanInTheLoopMiddleware` pauses a graph before specific tool calls execute, using LangGraph's `interrupt()`/`Command(resume=...)` mechanism underneath — genuinely relevant to every pattern above, since a handoff or a subagent consult can itself be exactly the kind of action worth gating on approval (imagine a specialist that can actually submit an expense reimbursement, not just look up the policy).

Requires a **checkpointer** — `interrupt()` works by pausing execution and persisting the paused state; there's nothing to resume *into* without somewhere to have saved it. `InMemorySaver` (Module 01 §5) is enough for this demo; production needs a persistent one (`AsyncPostgresSaver`, `MongoDBSaver`).

In [40]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

@tool
def submit_expense_reimbursement(amount: float, description: str) -> str:
    """Submit an expense for reimbursement. This actually processes a payment -- sensitive."""
    return f"Reimbursement of ${amount} submitted for: {description}"

guarded_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[retrieve_it_finance_policy, submit_expense_reimbursement],
    system_prompt="You answer IT/Finance questions and can submit expense reimbursements when asked.",
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"submit_expense_reimbursement": True},  # only the sensitive tool pauses
            description_prefix="Expense reimbursement pending approval",
        ),
    ],
    checkpointer=InMemorySaver(),
)
print("Human-in-the-loop agent constructed (construction only, no API call yet).")

Human-in-the-loop agent constructed (construction only, no API call yet).


**⚠️ Needs your API key** — this cell demonstrates the full pause/resume cycle. Ask it to submit a reimbursement; execution should stop with an interrupt *before* `submit_expense_reimbursement` actually runs. Confirmed against docs.langchain.com's current interrupts guide: what you're looking for specifically is a `'__interrupt__'` key in the printed `result` dict — that's the concrete signal execution actually paused, as opposed to the tool having quietly run to completion.

In [41]:
config = {"configurable": {"thread_id": "expense-demo"}}

result = guarded_agent.invoke(
    {"messages": [HumanMessage(content="Submit a $75 reimbursement for a client dinner.")]},
    config,
)
print(result)
# Expect result to contain an interrupt, NOT a completed tool call yet.

{'messages': [HumanMessage(content='Submit a $75 reimbursement for a client dinner.', additional_kwargs={}, response_metadata={}, id='85c91921-c8a5-44b2-8c47-b971aa02d6d6'), AIMessage(content=[{'text': "I'll submit that expense reimbursement for you right away!", 'type': 'text'}, {'id': 'toolu_017CoVjtwq1hH4gpBUpMGru4', 'caller': {'type': 'direct'}, 'input': {'amount': 75, 'description': 'Client dinner'}, 'name': 'submit_expense_reimbursement', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011Cd4gQTCDoeCfSiS1xoAmH', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 684, 'output_tokens': 91, 'output_tokens_details': None, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'cl

**⚠️ Needs your API key** — resume the paused run with an approval decision, using the *same* `thread_id` so it resumes the exact paused state above.

In [42]:
from langgraph.types import Command

resumed = guarded_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config,  # same thread_id -- this is what lets it find the paused state to resume
)
print(resumed["messages"][-1].content)

Your reimbursement has been successfully submitted! Here's a summary:

- **Amount:** $75.00
- **Description:** Client dinner
- **Status:** Submitted ✅

Keep an eye out for a confirmation and let me know if there's anything else I can help you with!


### 🔍 Under the hood — what actually paused, and why the checkpointer is load-bearing here

When `submit_expense_reimbursement` is about to run, `HumanInTheLoopMiddleware` intercepts it and calls `interrupt(...)` internally — this doesn't raise an exception or return normally, it **suspends the graph's execution entirely** at that exact point and returns control to you, with the paused state saved via whatever `checkpointer` was configured. `guarded_agent.invoke(...)` in the first cell above returns *before* the tool ever actually runs — that's the whole point, the approval gate sits strictly between "model decided to call this tool" and "tool actually executes."

`Command(resume={"decisions": [...]})` in the second cell doesn't start a new run — it looks up the *paused* run for that `thread_id` in the checkpointer and continues exactly where it left off, now with your decision available to the middleware. Reject instead of approve, and `submit_expense_reimbursement` never runs at all; the model sees a rejection message and responds accordingly. This is the concrete answer to "how do I stop an agent from taking a real-world action without a human checking first" — not a prompt instruction hoping the model asks first, a structural pause the model cannot skip.

## Shared vs. private state

Every agent above reads and writes the *same* `MessagesState` — full transparency, every specialist sees the whole conversation. Sometimes you want a specialist to have its own scratch space that the rest of the graph never sees (intermediate reasoning, draft citations, working numbers). LangGraph supports this via separate input/output/internal schemas per subgraph — the specialist's `PrivateState` fields simply aren't declared in the schema the parent graph passes state through, so they're invisible outside that subgraph by construction, not by convention.

In [43]:
from typing import TypedDict

class PublicState(TypedDict):
    question: str
    final_answer: str

class PrivateWorkingState(TypedDict):
    question: str
    draft_notes: str      # only this specialist's internal subgraph ever sees this
    final_answer: str

def draft_node(state: PrivateWorkingState) -> dict:
    # Private scratch work the parent graph will never see
    return {"draft_notes": f"considering how to answer: {state['question']}"}

def finalize_node(state: PrivateWorkingState) -> dict:
    return {"final_answer": f"Based on my notes ('{state['draft_notes']}'), here's the answer."}

specialist_builder = StateGraph(PrivateWorkingState, input_schema=PublicState, output_schema=PublicState)
specialist_builder.add_node("draft", draft_node)
specialist_builder.add_node("finalize", finalize_node)
specialist_builder.add_edge(START, "draft")
specialist_builder.add_edge("draft", "finalize")
specialist_builder.add_edge("finalize", END)

specialist_with_private_state = specialist_builder.compile()
result = specialist_with_private_state.invoke({"question": "What's the vacation policy?"})
print(result)  # only 'question' and 'final_answer' come back -- draft_notes never leaves the subgraph

{'question': "What's the vacation policy?", 'final_answer': "Based on my notes ('considering how to answer: What's the vacation policy?'), here's the answer."}


In [ ]:
print(result['messages'])
# why doesnt this work? Because stategraph is a prebuilt schema witth 'messages'?

KeyError: 'messages'

### 🔍 Under the hood — how `input_schema`/`output_schema` enforce privacy

`StateGraph(PrivateWorkingState, input_schema=PublicState, output_schema=PublicState)` tells LangGraph: internally, nodes can read/write anything in `PrivateWorkingState` (which includes `draft_notes`) — but what goes *in* to `.invoke()` and what comes *out* is filtered down to only the fields declared in `PublicState`. This isn't a convention you have to remember to follow; the schema itself is the enforcement mechanism — `draft_notes` structurally cannot leak into the parent graph's state, because the parent graph's `.invoke()` call and result are typed against `PublicState`, which never mentions it.

## Reflection / Reflexion — a genuinely different loop from ReAct

Contrast this carefully with everything so far. ReAct (Module 01) loops on the question *"do I need another tool call before I can answer?"* Reflection loops on a different question entirely: *"is my own finished answer actually good, or should I redo it?"* — the agent critiques its own output and revises, rather than gathering more information.

Tied to your `Summarize_Private_Documents_RAG` hallucination work: here's a generate → critique → (maybe regenerate) loop that catches an ungrounded RAG answer and fixes it, capped at a maximum number of revisions.

In [ ]:
from typing import TypedDict, Literal

class ReflectionState(TypedDict):
    question: str
    context: str
    draft_answer: str
    critique: str
    is_grounded: bool
    revision_count: int

def generate_node(state: ReflectionState) -> dict:
    critique_feedback = state["critique"]
    # why didnt this need a prompt template?
    prompt = (
        f"Context: {state['context']}\nQuestion: {state['question']}\n"
        + (f"Previous feedback to address: {critique_feedback}\n" if critique_feedback else "")
        + "Answer using ONLY the context above."
    )
    response = ChatAnthropic(model="claude-sonnet-4-6", max_tokens=200, temperature=0).invoke(prompt)
    return {"draft_answer": response.content}

def critique_node(state: ReflectionState) -> dict:
    critique_prompt = (
        f"Context: {state['context']}\nDraft answer: {state['draft_answer']}\n"
        "Does the draft answer ONLY use facts present in the context, with no invented "
        "details? Respond with exactly 'GROUNDED' or 'NOT GROUNDED: <what's wrong>'."
    )
    response = ChatAnthropic(model="claude-haiku-4-5", max_tokens=100, temperature=0).invoke(critique_prompt)
    verdict = response.content
    grounded = verdict.strip().startswith("GROUNDED")
    return {
        "is_grounded": grounded,
        "critique": "" if grounded else verdict,
        "revision_count": state["revision_count"] + 1,
    }

def route_after_critique(state: ReflectionState) -> Literal["generate", "__end__"]:
    if state["is_grounded"] or state["revision_count"] >= 2:  # hard cap -- never loop forever
        return "__end__"
    return "generate"

builder_reflection = StateGraph(ReflectionState)
builder_reflection.add_node("generate", generate_node)
builder_reflection.add_node("critique", critique_node)
builder_reflection.add_edge(START, "generate")
builder_reflection.add_edge("generate", "critique")
builder_reflection.add_conditional_edges("critique", route_after_critique, {"generate": "generate", "__end__": END})

reflection_graph = builder_reflection.compile()

result = reflection_graph.invoke({
    "question": "What are the vacation and expense policies?",
    "context": "Vacation: 15 days/year, rolls over up to 5. Expenses under $50 need no pre-approval.",
    "draft_answer": "", "critique": "", "is_grounded": False, "revision_count": 0,
})
print("Final answer:", result["draft_answer"])
print("Revisions needed:", result["revision_count"])

### 🔗 Ties back to theory — this is your hallucination-prevention notebook, turned into a loop instead of a single prompt

`Summarize_Private_Documents_RAG`'s fix for hallucination was a *static* prompt instruction ("say you don't know"). This is the *dynamic* version: generate an answer, have a second call specifically check it against the context, only accept it once it passes. Strictly more expensive (2+ LLM calls minimum) but catches failures the static instruction alone might miss — the two techniques compose rather than compete.

### 🔍 Under the hood — why `revision_count` capping matters, and `recursion_limit` as a backstop

Every loop needs an explicit exit condition or it can run forever — `route_after_critique`'s `revision_count >= 2` check is that exit condition here. As a safety net on top of your own loop logic, `.invoke()` accepts a `recursion_limit` (e.g. `graph.invoke(input, {"recursion_limit": 25})`) that hard-stops execution after that many graph steps regardless of your own conditions — worth setting on anything with a loop.

## Closing: Swarm, and which of the five patterns to actually reach for

**Swarm** — the one pattern named here but not built: the exact same `Command(goto=..., graph=Command.PARENT)` mechanism from §2, just with the central supervisor removed entirely. Agents hand off directly to whichever peer makes sense next, not always back through one hub. Lower latency (no round-trip through a supervisor on every step) at the real cost of harder debugging — no single place logs every routing decision, since there's no hub to log it. Default to a supervisor/hub topology (Handoffs, as built in §2) unless you have specific, measured evidence that supervisor round-trip latency is your actual bottleneck.

**The real decision guide, now that four of five are things you actually built rather than read about:**

| Pattern | Control | Memory across turns | Best fit |
|---|---|---|---|
| **Router** (§1) | One-shot classify, then hands off entirely | None — stateless per call | Clear input categories, lightweight/deterministic classification |
| **Handoffs** (§2) | Genuinely transfers; specialist takes over | Whatever the specialist's own state carries | A different agent should *become* the one talking — this notebook's HR/IT split |
| **Subagents** (§3) | Stays with the main agent; specialists report back | Main agent maintains context across turns | Self-contained sub-tasks, especially ones that can run in parallel |
| **Skills*** (§4) | Never leaves the one agent | Same agent, same context, just more of it loaded | One job, several modes — cheaper than any multi-agent option |
| **Swarm** (not built) | Transfers peer-to-peer, no hub | Same as Handoffs, decentralized | Latency-critical, and you've already measured Handoffs' supervisor round-trip as the bottleneck |

\* Skills isn't actually multi-agent — no second agent exists — it's in this table because it answers the same question as the other four, not because it belongs in the same technical category.

Default to the cheapest pattern that actually solves your problem — Skills before Subagents, Subagents before Handoffs, Handoffs before Swarm — and only reach for Human-in-the-loop (§5) the moment any of the above can take an action you wouldn't want happening unsupervised.